# PySpark 8: real-world use cases and interviews

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-08-real-world-and-interviews.ipynb)

Use these design reviews to choose a pipeline for a business requirement. For each case, identify the input, output grain, failure you expect to recover from, and the evidence you would use to declare the job healthy.

Read the examples as production sketches. The cluster, storage paths, and integrations depend on your platform. For an executable pipeline that covers ingestion through replay, use the linked capstone.

## Choose a pipeline design for the workload

### 1. Nightly batch ELT into a partitioned lake

The nightly delivery is a full snapshot for one business date. Read the date from the job
parameters and validate that every row belongs to that date before publishing. The example
writes a candidate to a run-specific staging path so a failed check leaves the published
data available to readers.

```python
business_date = "2026-10-04"
orders = spark.read.schema(order_schema).json(
    f"s3://acme-raw/orders/ds={business_date}/"
)
# These checks raise if the contract is broken.
validate_delivery(orders, expected_date=business_date)
orders.write.mode("errorifexists").parquet(
    f"s3://acme-staging/orders/run_id={run_id}/"
)
```

Reconcile the staged row count and amount against the delivery manifest before replacing
that date in the target. On a transactional table, scope the replacement to that date.
If the delivery contains only changes, merge by order key and source version instead.

A complete-day replacement and a partial change delivery need different write rules. Record the delivery ID, target date, and committed table version so an operator can tell what happened after a retry.

### 2. Structured Streaming from Kafka into a lakehouse table

A streaming job decodes Kafka events, applies the batch transformations, and merges
micro-batches into an Iceberg or Delta table:

```python
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .writeStream
    .foreachBatch(merge_into_lakehouse)          # idempotent MERGE per batch
    .option("checkpointLocation", "s3://acme-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)
```

The checkpoint records which offsets Spark has processed. The callback can run again for the same batch, so the merge must use stable keys and source versions. Route malformed events to quarantine with their topic, partition, and offset so they can be investigated and replayed.

### 3. Feature engineering for machine learning

Feature tables use broadcast joins to small dimensions and built-in column expressions. A z-score
needs the mean and standard deviation of the whole column, so compute them once with an aggregate and
broadcast them back:

```python
spend = (
    events.join(F.broadcast(products), "product_id")
    .groupBy("customer_id")
    .agg(F.count("*").alias("orders_30d"), F.sum("amount").alias("spend_30d"))
)
stats = spend.agg(F.mean("spend_30d").alias("mu"), F.stddev("spend_30d").alias("sigma"))
features = (
    spend.crossJoin(F.broadcast(stats))
    .withColumn("spend_z", (F.col("spend_30d") - F.col("mu")) / F.col("sigma"))
    .drop("mu", "sigma")
)
```

A Series-to-Series `pandas_udf` such as `lambda s: (s - s.mean()) / s.std()` looks equivalent but is
wrong: Spark calls it once per Arrow batch, so `s.mean()` is the mean of that batch, not of the column.
Keep pandas UDFs for row-local logic that has no built-in function.

Why Spark: the same feature definitions can be applied to historical training data and to daily scoring runs, which reduces train/serve skew. Save the training `mu` and `sigma` and reuse them at scoring time; recomputing them on each scoring batch is itself a common source of skew.

### 4. Log and event processing with messy input

Semi-structured logs break schema assumptions. Read permissively (as in the corrupt-record
example in [reading and cleaning](pyspark-02-reading-sql-and-cleaning.ipynb)), write good rows to the curated table and bad rows to a quarantine path, and evolve
the schema deliberately.

Why Spark: quarantine plus schema-on-read lets ingestion continue while the platform team fixes the producer, instead of failing the whole day.

### 5. Backfills and reprocessing without duplicating data

A bug fix requires reprocessing September. Set
`spark.sql.sources.partitionOverwriteMode=dynamic` and overwrite with only September's rows: the
dynamic-overwrite cell in PySpark 5 shows exactly this on one day.

Dynamic overwrite replaces partitions represented in the output. Verify that the replacement contains all rows for those dates. Handle dates whose correct result is empty explicitly, since they will have no output rows to trigger replacement.

## What should you say in a PySpark interview answer?

A strong PySpark interview answer should include the API, the execution model, and the production concern.

Example answer for "How do you deduplicate orders?": `row_number()` over
`Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc())`, keep `rn = 1` (the CDC
example in [reading and cleaning](pyspark-02-reading-sql-and-cleaning.ipynb)), then explain:

+ `row_number` keeps one row per business key.
+ Ordering must use source metadata, not random ingestion order.
+ The result should be merged into the target for idempotency.
+ If one key is extremely hot, this window can skew and needs tuning.

How it works under the hood, restated as an answer structure. A strong answer names the API, the execution model, and the failure mode. For deduplication: "I use `row_number()` over `Window.partitionBy(business_key).orderBy(source_lsn desc)` filtered to `rn = 1`, then `MERGE`; that forces one shuffle by key, and if one key is hot the window skews, so I check the partition sizes and salt or let AQE skew join split it."

Worked example, a 30-second answer skeleton:

```text
1. API       - the exact functions and window spec you call
2. Engine    - narrow vs wide, where the shuffle is, partition count
3. Ops       - idempotency, skew, cost, monitoring, failure recovery
```

**Watch out for.** answering with syntax only. Interviewers probe the plan, so tie every method to a shuffle, a partition count, or a failure mode.

## Production case: replay a late orders delivery

A **backfill** rebuilds a historical slice from retained input. Start by naming the affected business dates and determining whether each delivery contains a full replacement or a set of changes. That determines whether to overwrite the slice or merge its changed records.

**Where it matters.** A vendor delivers Monday's missing orders on Wednesday. Rebuild Monday's silver data and gold totals while keeping Tuesday's results intact.

**Watch out for.** Check whether other jobs can write the same slice concurrently. Preserve the original delivery, and use business dates for the affected partitions so a late delivery does not end up in Wednesday's reporting data.

### Example

Monday originally had 98 orders totaling 490000 cents. The late delivery contains two new orders worth 10000 cents in total. After the replay, Monday should have 100 orders totaling 500000 cents, Tuesday should retain its previous totals, and running the replay twice should leave the same result.

## Practice: PySpark concepts

The hands-on practice for this topic lives in its own notebook: [PySpark concepts, end to end](pyspark-concepts-end-to-end.ipynb) — [Open in Colab](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-concepts-end-to-end.ipynb).

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Name the API, the execution model and the cost before you name the code.
- Backfills reuse the daily job with a date parameter and dynamic overwrite.
- Practise the answer structure on the interview notebooks in this handbook.